# 2.1 LSTM ablations

**Owner:** Winston Ho Rui Wei · **Stage 1:** personalised relevance shortlist.

The main run (`20261104_014914_seed42`) ranks the full time-eligible catalogue
with an LSTM and target-aware attention. This notebook tests which parts of that
model earn their cost for the top-100 shortlist, starting with whether it beats
no-training baselines. Every trained arm starts from the main run's tuned
settings and changes **one** thing. The primary decision metric is catalogue
known-click Recall@100.

All numbers are validation development results on two fixed samples that
exclude the final rerun's 2,000-request selection sample. The test split is
never loaded. Ablation arms are diagnostics, not handoff models, so the standard
export and reload steps do not apply.

**How to run.** Run section 1 (setup), then any single section. Each section
saves its results to `DATA_ROOT/Experiments/2.1_lstm/ablation/<RUN_ID>/arms/` and
reads nothing another section left in memory. A rerun with the same `RUN_ID`
skips arms whose `metrics.json` exists and reuses a saved `model_state.pt`.
Sections are ordered by importance; delete from the bottom up. The results
section reads whatever is saved and also runs after setup alone.

| § | Section | Group | Training | Est. L4 time |
| --- | --- | --- | --- | --- |
| 1 | Setup: config, data, samples, arm specs, helpers | – | no | ~10 min |
| 2 | Most recent and time-aware popularity | A | no | ~2 min |
| 3 | Main-run model, and recency → LSTM rerank (N = 200, 500, 1000) | A | no | ~3 min |
| 4 | Full model × 3 seeds (noise floor) | Seeds | 3 fits | ~10 min |
| 5 | `last_state` × 3 and `attention_no_lstm` × 3 | B | 6 fits | ~18 min |
| 6 | `metadata_only` (no vectors, no availability flags) | D | 1 fit | ~3 min |
| 7 | `no_history` | D | 1 fit | ~3 min |
| 8 | Logistic regression (recorded slates only) | A | CPU fit | ~5 min |
| 9 | Evaluation-time history permutation (main-run model) | C | no | ~2 min |
| 10 | `no_text`, `no_image` | D | 2 fits | ~6 min |
| 11 | Seed ensemble (reuses section 4 weights, else trains them) | Seeds | 0–3 fits | ~4 min |
| 12–20 | Tier 2: history 10, history 20, no read features, ID embedding, no text dropout, train-time shuffle, no recency, 25% and 50% of training requests | C, D, E | 1 fit each | ~3 min each |
| 21–25 | Results, segments, vector share, training curves, reading template | – | no | ~2 min |

Timings come from the main run on an L4: about 25 s per training epoch and about
14 ms per request to score a full catalogue of about 124,000 articles. One fit is
4 epochs (about 100 s) plus about 80 s of catalogue scoring on 5,000 requests.

| Group | Arms | Lecture source |
| --- | --- | --- |
| A. Baselines | most recent, time-aware popularity, recency → LSTM rerank, logistic regression (recorded slates only) | Week 3 "build a robust baseline"; Week 8 notes "no baseline, so you cannot tell how much attention adds"; Week 11 "no popularity baseline" |
| B. History summary | `last_state`, `attention_no_lstm` | Week 7 sequence classification (slide 24); Week 8 target-aware attention (slides 16–17) |
| C. Order and length | evaluation-time history permutation, train-time shuffle, history length 10/20, no recency term | Week 7 RNN memory; Week 8 sequence-length slides 19–21 |
| D. Inputs and cold start | metadata only (no vectors and no availability flags), no history, no text vectors, no image vectors, vector-share check, no per-event read features, ID embedding on, no text dropout | Week 6 embeddings, fusion and ID masking; Week 3 "stable features vs ID proxies"; Week 10 leave-one-group-out / leakage-safe features |
| Seeds and ensemble | full model × 3 seeds, 3-seed ensemble | Week 3 variability; Week 11/12 seed repeats; Week 12 bagging (variance floor ρσ²) |
| E. Training data size | learning curve at 25% and 50% of training requests | No lecture link assigned; a data-budget diagnostic |
| Protocol | one change per arm; validation only | Week 3/12 "change one thing at a time", "never select on test" |

### Pre-registered decision rules

These rules are fixed before any arm runs. The results section applies them
mechanically; the reading template uses them.

1. **Primary metric.** Catalogue known-click Recall@100 on the 5,000-request
   catalogue sample, full eligible pool per request. Observed-click nDCG@100,
   recorded-slate mean impression AUC, slate nDCG@5 and slate MRR are secondary
   and descriptive. Parameters, training time and scoring latency are costs.
2. **Difference.** Δ = arm − reference, per request. The reference is the full
   model unless stated: a 3-seed arm uses seed-matched pairs and reports the mean
   Δ over seeds; a single-seed arm and most comparators pair with the full
   model's seed 42; the ensemble pairs with the mean of the three single full
   models. Arms that reuse the saved main-run model pair with it
   (`permute_eval`) or with `most_recent` (the rerank arms).
3. **Interval.** 95% paired user-cluster bootstrap (`user_cluster_bootstrap`,
   2,000 resamples, bootstrap seed 0); users are the resampling unit.
4. **Noise floor.** The seed spread of the 3-seed full model on Recall@100
   (max − min of the three sample means).
5. **Decision.** An arm counts as different from its reference only if its CI
   excludes 0 **and** |Δ| exceeds the seed spread. Otherwise the result is
   "no detectable difference", not "equal".
6. **Baseline headline.** If the full model's Recall@100 is below the
   most-recent comparator, that is the headline result. Component arms are then
   read only relative to the full model (what moves it toward or away from the
   baseline), not as evidence that a component earns business value.
7. **LSTM on top of recency.** The LSTM adds value as a ranker of recency
   candidates only if a rerank arm beats `most_recent` by rule 5. Each rerank
   arm's recall ceiling (Recall@N of the newest N articles) bounds what it can reach.
8. **Attention.** Attention earns its cost only if the full model beats
   `last_state` by more than the seed spread with a CI excluding 0. The LSTM
   earns its cost against `attention_no_lstm` by the same rule.
9. **Long histories.** The long-history framing is used only if the segment
   table shows it: the full model's advantage over `last_state` in the 50+
   event buckets must satisfy rule 5 within those buckets and exceed its
   advantage in the shorter buckets.
10. **Order.** The model uses event order only if evaluation-time permutation
    of the validation histories lowers the main-run model's Recall@100 by rule 5.
11. **Exposure shortcut.** Frozen vectors exist only for articles in this
    dataset's logs, so vector availability can mark likely click targets. The
    vector-share table compares the vector-less share of the full model's top 100
    with the fresh pool and with `metadata_only`, which has no vectors and no
    availability flags. The full model's Δ against `metadata_only` is judged by
    rule 5; it bounds what the vectors and that availability signal add together.
12. **Scope.** The logistic regression is a recorded-slate-only comparison; it
    supports no catalogue claim. Validation only; test is never loaded, and the
    exclusion set never contributes to any reported number. Tier 2 arms use the
    same rules and are exploratory.

## 1. Setup

In Colab, choose the L4 GPU runtime and upload **`lstm_data.py`, `lstm_model.py`
and `ablation_variants.py` beside this notebook in `/content`**. Locally, keep
the three helpers beside the notebook.

Hyperparameters, fitted preprocessing and the frozen article inputs load from
the main run's bundle, so every arm sees exactly the main run's inputs. The saved
article table is the shared union catalogue, so it also defines the eligible
pool. The main run's `training_history.json` justifies the fixed budget:
Recall@100 improved at every epoch through epoch 4 (the selected checkpoint),
and the learning rate stayed constant through epoch 5. `ReduceLROnPlateau`
(patience 0) first halved it after epoch 5, so it never fired within 4 epochs.
Every trained arm therefore runs exactly 4 epochs at the tuned constant learning
rate, with no early stopping and no per-epoch selection evaluation.

**Samples.** The final rerun will select on the first 2,000 validation requests
of the shared BLAKE2b order (`seed 42`, label `validation`); that exclusion set
never contributes to a reported number. The **catalogue sample** is 5,000 of the
remaining requests by the same BLAKE2b-64 rule with the label
`validation_ablation_catalogue` in place of the split name, processed
chronologically; each request scores its full eligible pool. The
**recorded-slate sample** is every validation request outside the exclusion set.

In [ ]:
from pathlib import Path

EXPERIMENT_ID = "2.1_lstm"
OWNER = "Winston Ho Rui Wei"
DATA_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder")
MAIN_RUN_ID = "20261104_014914_seed42"   # tuned main run; hyperparameters and saved model load from it
RUN_ID = "ablation_v1"                   # same ID = resume; change RUN_ID whenever the protocol changes
SEEDS = (42, 43, 44)                     # full model and group B; other arms use SEEDS[0]
EPOCHS = 4                               # fixed budget, constant learning rate, no early stopping

SAMPLE_SEED = 42                         # request-sampling seed (independent of model seeds)
EXCLUSION_REQUESTS = 2000                # final rerun's selection sample; never reported on
SELECTION_REQUESTS = 100                 # main run's selection sample (prefix check only)
CATALOGUE_REQUESTS = 5000
CATALOGUE_SAMPLE_LABEL = "validation_ablation_catalogue"   # BLAKE2b salt replacing the split name
K = 100                                  # selection cutoff (Recall@100, nDCG@100)
N_BOOT, BOOT_SEED = 2000, 0
CATALOGUE_CHUNK = 32768

In [ ]:
import sys, json, time, importlib.util
from dataclasses import asdict, fields, replace

if "google.colab" in sys.modules or Path("/content").is_dir():
    from google.colab import drive
    drive.mount("/content/drive")
    CODE_DIR = Path("/content")
else:
    CODE_DIR = Path.cwd()
sys.path.insert(0, str(CODE_DIR))
for name in ("lstm_data", "lstm_model", "ablation_variants"):
    sys.modules.pop(name, None)
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch
import matplotlib.pyplot as plt
import lstm_data as D
import lstm_model as M
import ablation_variants as V

MAIN_RUN_DIR = DATA_ROOT / "Experiments" / EXPERIMENT_ID / MAIN_RUN_ID
ABLATION_DIR = DATA_ROOT / "Experiments" / EXPERIMENT_ID / "ablation" / RUN_ID
ARMS_DIR = ABLATION_DIR / "arms"
ARMS_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MAIN_SEED = SEEDS[0]
ENSEMBLE_LABEL = "_".join(map(str, SEEDS))

def log(*parts):
    print(*parts, flush=True)

RUN_CONFIG = json.loads((MAIN_RUN_DIR / "run_config.json").read_text())
MODEL_CFG = M.ModelConfig.from_json(json.loads((MAIN_RUN_DIR / "model" / "model_config.json").read_text()))
PREP = json.loads((MAIN_RUN_DIR / "model" / "preprocessing.json").read_text())
MAIN_HISTORY = json.loads((MAIN_RUN_DIR / "training_history.json").read_text())
TRAIN_CFG = M.TrainConfig(**RUN_CONFIG["train_config"])
log("Ablation run", RUN_ID, "| device", DEVICE, "| results:", ABLATION_DIR)
log("Main run by epoch: Recall@100", [r["known_click_recall@100"] for r in MAIN_HISTORY],
    "| learning rate", [f"{r['learning_rate']:.2e}" for r in MAIN_HISTORY])
log("Tuned model:", {k: v for k, v in MODEL_CFG.to_json().items()
                     if k in ("variant", "lstm_hidden", "attention_dim", "article_dim", "dropout", "text_dropout")})
log("Tuned training:", {k: getattr(TRAIN_CFG, k) for k in
                        ("learning_rate", "weight_decay", "history_dropout", "batch_impressions", "grad_clip")})

In [ ]:
PATHS, _ = D.resolve_paths(DATA_ROOT, required=["features_train", "features_validation", "history_train",
                                                "history_validation", "behaviors_train", "behaviors_validation"])
FRAMES, OFFSETS, SPLIT_START = {}, {}, {}
for split in ("train", "validation"):               # the test split is never loaded
    frame, info = D.load_candidate_table(PATHS[f"features_{split}"], split)
    FRAMES[split], OFFSETS[split] = frame, info["offsets"]
    SPLIT_START[split] = pd.Timestamp(frame["impression_time"].min())
HISTORIES = {s: D.build_history(PATHS[f"history_{s}"], SPLIT_START[s], PREP["history_max_len"])
             for s in FRAMES}

with np.load(MAIN_RUN_DIR / "model" / "article_inputs.npz", allow_pickle=False) as saved:
    article_fields = {f.name: saved[f.name] for f in fields(D.ArticleTable)}
article_fields["numeric_names"] = article_fields["numeric_names"].tolist()
ARTICLES = D.ArticleTable(**article_fields)
NUMERIC_NAMES = list(ARTICLES.numeric_names)

ARRAYS = {s: D.build_split_arrays(s, FRAMES[s], OFFSETS[s], HISTORIES[s], ARTICLES,
                                  PREP["numeric_stats"], PREP["device_columns"]) for s in FRAMES}
VAL = ARRAYS["validation"]
EVENT_NAMES = list(VAL.event_names)
INDEX = D.build_catalogue_index("all", "union", ARTICLES.article_ids[1:], ARTICLES)
CLICKS_VAL = D.clicked_eligibility(VAL, INDEX, ARTICLES)
CLICK_LOOKUP = D.clicks_by_request(CLICKS_VAL)
AGE_STATS = PREP["numeric_stats"]["log_candidate_age_h"]

EXCLUSION, EXCLUSION_BIDS = D.select_requests(VAL.user_id, VAL.impression_id, VAL.imp_time_us,
                                              "validation", SAMPLE_SEED, EXCLUSION_REQUESTS)
SELECTION, SELECTION_BIDS = D.select_requests(VAL.user_id, VAL.impression_id, VAL.imp_time_us,
                                              "validation", SAMPLE_SEED, SELECTION_REQUESTS)
main_selection = pd.read_parquet(MAIN_RUN_DIR / "selection" / "validation_requests.parquet")["behaviour_id"].tolist()
REMAINING = np.setdiff1d(np.arange(VAL.n_impressions), EXCLUSION)
CAT_SAMPLE, CAT_BIDS = V.hash_sample(VAL.user_id, VAL.impression_id, VAL.imp_time_us,
                                     CATALOGUE_SAMPLE_LABEL, SAMPLE_SEED, CATALOGUE_REQUESTS, among=REMAINING)
SLATE_SAMPLE = REMAINING
SLATE_BIDS = D.behaviour_ids(VAL.user_id[SLATE_SAMPLE], VAL.impression_id[SLATE_SAMPLE]).tolist()
CAT_USERS, SLATE_USERS = VAL.user_id[CAT_SAMPLE], VAL.user_id[SLATE_SAMPLE]
SLATE_ARRAYS = D.subset_split_arrays(VAL, SLATE_SAMPLE)
lo, hi = INDEX.bounds(VAL.imp_time_us[CAT_SAMPLE])
CAT_PAIRS = int((hi - lo).sum())
log("Requests/candidates:", {s: (a.n_impressions, a.n_candidates) for s, a in ARRAYS.items()})
log("Selection sample inside the exclusion set:", set(SELECTION_BIDS) <= set(EXCLUSION_BIDS),
    "| equals the main run's saved selection IDs:", main_selection == SELECTION_BIDS)
log("Catalogue sample:", len(CAT_BIDS), "requests,", CAT_PAIRS, "eligible pairs | recorded slates:",
    len(SLATE_BIDS), "requests,", SLATE_ARRAYS.n_candidates, "candidates")

In [ ]:
S3, S1 = list(SEEDS), [MAIN_SEED]
ARMS = {   # trained arms: each changes one thing relative to "full"; a section adds any input only it needs
    "full": dict(tier=1, group="Seeds", seeds=S3, change="none: tuned main model under the fixed 4-epoch budget"),
    "last_state": dict(tier=1, group="B", seeds=S3, model=dict(variant="last_state"),
                       change="history summary = final LSTM state (no target-aware attention)"),
    "attention_no_lstm": dict(tier=1, group="B", seeds=S3, model=dict(variant="attention_no_lstm"),
                              change="attention over per-event inputs (no LSTM)"),
    "metadata_only": dict(tier=1, group="D", seeds=S1,
                          ablation=dict(use_text_vectors=False, use_image_vectors=False,
                                        drop_article_numeric=V.VECTOR_FLAG_COLUMNS),
                          change="both vector blocks and all three availability flags removed"),
    "no_history": dict(tier=1, group="D", seeds=S1, data="no_history",
                       change="history removed: empty-history path and context for every request"),
    "no_text": dict(tier=1, group="D", seeds=S1, ablation=dict(use_text_vectors=False),
                    change="text vector block removed (has_text flag kept)"),
    "no_image": dict(tier=1, group="D", seeds=S1, ablation=dict(use_image_vectors=False),
                     change="image vector block removed (has_image flag kept)"),
    "history_10": dict(tier=2, group="C", seeds=S1, data="history_10", change="most recent 10 history events (of 50)"),
    "history_20": dict(tier=2, group="C", seeds=S1, data="history_20", change="most recent 20 history events (of 50)"),
    "no_read_features": dict(tier=2, group="D", seeds=S1, ablation=dict(drop_event_features=V.READ_EVENT_COLUMNS),
                             change="per-event read time and scroll removed"),
    "id_embedding": dict(tier=2, group="D", seeds=S1, change="learned article-ID embedding on"),
    "no_text_dropout": dict(tier=2, group="D", seeds=S1, model=dict(text_dropout=0.0),
                            change="training text dropout 0.2 -> 0"),
    "train_shuffled": dict(tier=2, group="C", seeds=S1, data="shuffled", model=dict(variant="attention_shuffled"),
                           change="histories in a fixed random order (train and validation)"),
    "no_recency": dict(tier=2, group="C", seeds=S1, ablation=dict(use_recency=False),
                       change="recency term removed from attention"),
    "train_25pct": dict(tier=2, group="E", seeds=S1, change="25% of training requests"),
    "train_50pct": dict(tier=2, group="E", seeds=S1, change="50% of training requests"),
}
COMPARATORS = {   # no training; "reference" = full seeds to pair with, or another arm's name
    "most_recent": dict(tier=1, group="A", seeds=S1, reference=S1,
                        change="no model: eligible catalogue by publication time, newest first"),
    "popularity_24h": dict(tier=1, group="A", seeds=S1, reference=S1,
                           change="no model: clicks in the trailing 24 h, ties by recency"),
    "main_model": dict(tier=1, group="A", seeds=S1, reference=S1,
                       change="saved main-run model (no retraining) on the full catalogue"),
    **{f"rerank_{n}": dict(tier=1, group="A", seeds=S1, reference="most_recent",
                           change=f"newest {n} eligible articles, reranked by the main-run model")
       for n in (200, 500, 1000)},
    "logistic_regression": dict(tier=1, group="A", seeds=S1, reference=S1,
                                change="Baseline Testing logistic regression; recorded slates only"),
    "permute_eval": dict(tier=1, group="C", seeds=S1, reference="main_model",
                         change="main-run model, validation histories permuted at evaluation"),
    "ensemble": dict(tier=1, group="Seeds", seeds=[ENSEMBLE_LABEL], reference=S3,
                     change="mean logit of the three full-model seeds"),
}

ARTICLE_TENSORS = M.article_table_tensors(ARTICLES, DEVICE)
BASE_TENSORS = {"train": M.SplitTensors(ARRAYS["train"], DEVICE), "val": M.SplitTensors(VAL, DEVICE),
                "slates": M.SplitTensors(SLATE_ARRAYS, DEVICE)}

In [ ]:
def arm_dir(name, seed):
    return ARMS_DIR / f"{name}_seed{seed}"

def done(name, seed):
    return (arm_dir(name, seed) / "metrics.json").is_file()

def cuda_sync():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

def free_gpu():
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

def latency(catalogue_summary):
    seconds = catalogue_summary["scoring_seconds"]
    return {"catalogue_ms_per_request": 1e3 * seconds / len(CAT_SAMPLE),
            "catalogue_ms_per_1k_candidates": 1e6 * seconds / CAT_PAIRS}

def save_run(name, seed, metrics, catalogue=None, slates=None):
    out = arm_dir(name, seed)
    out.mkdir(parents=True, exist_ok=True)
    if catalogue is not None:
        V.catalogue_frame(CAT_SAMPLE, CAT_BIDS, CAT_USERS, catalogue).to_parquet(out / "catalogue.parquet", index=False)
    if slates is not None:
        V.slate_frame(SLATE_SAMPLE, SLATE_BIDS, SLATE_USERS, slates).to_parquet(out / "slates.parquet", index=False)
    D.write_json({"arm": name, "seed": seed, **metrics}, out / "metrics.json")   # last: marks completion
    log(f"{name} seed {seed} saved | Recall@100 {(metrics.get('catalogue') or {}).get('known_click_recall@100')} "
        f"| slate AUC {(metrics.get('recorded_slates') or {}).get('mean_impression_auc')}")

def evaluate_catalogue(model, val_tensors, members=None):
    models = members or [model]
    t0 = time.time()
    scorers = [M.CatalogueScorer(m, val_tensors, INDEX, AGE_STATS, chunk=CATALOGUE_CHUNK, requests=CAT_SAMPLE)
               for m in models]
    cuda_sync()
    setup = time.time() - t0
    if members:
        topk, tie_fn = V.EnsembleTopK(scorers).topk, None
    else:
        topk, tie_fn = scorers[0].topk, (lambda: scorers[0].last_tie_at_cutoff)
    cat, cat_per = V.evaluate_catalogue(topk, CAT_SAMPLE, CLICKS_VAL, K, tie_fn=tie_fn, sync=cuda_sync)
    return cat, cat_per, {"scorer_setup_seconds": setup, **latency(cat)}

def evaluate_slates(models, slate_tensors):
    t0 = time.time()
    scores = np.mean([M.predict_logits(m, slate_tensors, batch_size=TRAIN_CFG.eval_batch_impressions)
                      for m in models], axis=0)
    slate, slate_per = V.slate_metrics(SLATE_ARRAYS.offsets, SLATE_ARRAYS.labels, scores)
    return slate, slate_per, time.time() - t0

def evaluate_model(model, val_tensors, slate_tensors, members=None):
    """Catalogue sample (exact full-pool top 100) and recorded-slate sample."""
    cat, cat_per, timing = evaluate_catalogue(model, val_tensors, members)
    slate, slate_per, timing["slate_scoring_seconds"] = evaluate_slates(members or [model], slate_tensors)
    free_gpu()
    return cat, cat_per, slate, slate_per, timing

def load_main_model():
    """The saved main-run model, reloaded as experiment.ipynb's reload check does (bundled code + config)."""
    spec = importlib.util.spec_from_file_location("saved_lstm_model", MAIN_RUN_DIR / "model" / "code" / "lstm_model.py")
    BM = importlib.util.module_from_spec(spec)
    sys.modules["saved_lstm_model"] = BM
    spec.loader.exec_module(BM)
    cfg = BM.ModelConfig.from_json(json.loads((MAIN_RUN_DIR / "model" / "model_config.json").read_text()))
    model = BM.build_model(cfg, ARTICLE_TENSORS, DEVICE)
    model.load_state_dict(torch.load(MAIN_RUN_DIR / "model" / "model_state.pt", map_location=DEVICE, weights_only=True))
    return model.eval()

def arm_arrays(spec):
    """(train, validation) SplitArrays for a trained arm."""
    data, pair = spec.get("data"), (ARRAYS["train"], VAL)
    if data == "no_history":
        pair = tuple(V.remove_history(a, PREP["numeric_stats"]) for a in pair)
    elif data in ("history_10", "history_20"):
        pair = tuple(V.truncate_history(a, int(data.split("_")[1]), EVENT_NAMES) for a in pair)
    elif data == "shuffled":
        pair = tuple(D.permute_history(a, spec["permute_seed"]) for a in pair)
    train, val = pair
    if spec.get("train_requests") is not None:
        train = D.subset_split_arrays(train, spec["train_requests"])
    return train, val

def arm_model(name, seed):
    spec = ARMS[name]
    cfg = replace(MODEL_CFG, **spec.get("model", {}))
    acfg = V.AblationConfig(**spec.get("ablation", {}))
    M.set_seed(seed)
    model = V.build_arm_model(cfg, acfg, spec.get("article_tensors", ARTICLE_TENSORS), DEVICE,
                              NUMERIC_NAMES, EVENT_NAMES)
    return model, cfg, acfg

def load_arm_model(name, seed):
    model, _, _ = arm_model(name, seed)
    model.load_state_dict(torch.load(arm_dir(name, seed) / "model_state.pt", map_location=DEVICE, weights_only=True))
    return model.eval()

def selection_check(model):
    """Reproducibility only: Recall@100 on the main run's selection sample (inside the exclusion set)."""
    scorer = M.CatalogueScorer(model, BASE_TENSORS["val"], INDEX, AGE_STATS, chunk=CATALOGUE_CHUNK, requests=SELECTION)
    shortlists, _ = M.catalogue_shortlists(scorer, SELECTION, K)
    return {"sample": "main-run 100-request selection sample (exclusion set; not a reported result)",
            "recall@100": D.known_click_recall(shortlists, CLICKS_VAL, (K,))["known_click_recall@100"],
            "main_run_epoch4_recall@100": MAIN_HISTORY[EPOCHS - 1]["known_click_recall@100"]}

def run_trained_arm(name, seed):
    """Skip if metrics.json exists; else reuse model_state.pt if present; else train. Then evaluate and save."""
    if done(name, seed):
        log(f"skip {name} seed {seed}: complete")
        return
    spec = ARMS[name]
    tensors = BASE_TENSORS
    if spec.get("data") or spec.get("train_requests") is not None:
        train_arrays, val_arrays = arm_arrays(spec)
        tensors = {"train": M.SplitTensors(train_arrays, DEVICE), "val": BASE_TENSORS["val"],
                   "slates": BASE_TENSORS["slates"]}
        if val_arrays is not VAL:
            tensors.update(val=M.SplitTensors(val_arrays, DEVICE),
                           slates=M.SplitTensors(D.subset_split_arrays(val_arrays, SLATE_SAMPLE), DEVICE))
    model, cfg, acfg = arm_model(name, seed)
    out = arm_dir(name, seed)
    out.mkdir(parents=True, exist_ok=True)
    state = out / "model_state.pt"
    if state.is_file():
        model.load_state_dict(torch.load(state, map_location=DEVICE, weights_only=True))
        model.eval()
        train_history = json.loads((out / "train_history.json").read_text())
        log(f"{name} seed {seed}: reusing saved weights")
    else:
        tcfg = replace(TRAIN_CFG, seed=seed, max_epochs=EPOCHS)   # early-stopping fields are unused
        log(f"{name} seed {seed}: training {EPOCHS} epochs on {tensors['train'].n_impressions} requests")
        train_history = V.train_fixed_budget(model, tensors["train"], tcfg, EPOCHS, log)
        D.write_json({"arm": name, "seed": seed, "change": spec["change"], "model_config": cfg.to_json(),
                      "ablation": acfg.to_json(), "train_config": asdict(tcfg), "data": spec.get("data"),
                      "train_requests": tensors["train"].n_impressions}, out / "arm_config.json")
        D.write_json(train_history, out / "train_history.json")
        torch.save(model.state_dict(), state)              # written only after training completes
    cat, cat_per, slate, slate_per, timing = evaluate_model(model, tensors["val"], tensors["slates"])
    metrics = {"tier": spec["tier"], "group": spec["group"], "change": spec["change"], "kind": "trained",
               "parameters": M.count_parameters(model)["trainable_parameters"],
               "train_seconds": float(sum(h["seconds"] for h in train_history)),
               "train_history": train_history, "catalogue": cat, "recorded_slates": slate, "timing": timing}
    if name == "full" and seed == MAIN_SEED:
        metrics["reproducibility_check"] = selection_check(model)
        log("Reproducibility check:", metrics["reproducibility_check"])
    save_run(name, seed, metrics, cat_per, slate_per)
    del model, tensors
    free_gpu()

def comparator_metrics(name, catalogue, slate, timing, parameters, train_seconds=0.0, **extra):
    spec = COMPARATORS[name]
    return {"tier": spec["tier"], "group": spec["group"], "change": spec["change"], "kind": "no training",
            "parameters": parameters, "train_seconds": train_seconds, "catalogue": catalogue,
            "recorded_slates": slate, "timing": timing, **extra}

def run_ranker(name, ranker, **extra):
    """No-model comparator scored on both samples (``RecencyPopularityRanker``)."""
    if done(name, MAIN_SEED):
        log(f"skip {name}: complete")
        return
    cat, cat_per = V.evaluate_catalogue(lambda i, k: ranker.topk(int(VAL.imp_time_us[i]), k),
                                        CAT_SAMPLE, CLICKS_VAL, K)
    t0 = time.time()
    scores = ranker.slate_scores(SLATE_ARRAYS.cand_article_row,
                                 np.repeat(SLATE_ARRAYS.imp_time_us, np.diff(SLATE_ARRAYS.offsets)))
    slate, slate_per = V.slate_metrics(SLATE_ARRAYS.offsets, SLATE_ARRAYS.labels, scores)
    timing = {**latency(cat), "slate_scoring_seconds": time.time() - t0}
    save_run(name, MAIN_SEED, comparator_metrics(name, cat, slate, timing, 0, **extra), cat_per, slate_per)

## 2. Most recent and time-aware popularity (no training)

* **Most recent:** the eligible catalogue by publication time, newest first, then
  ascending article ID; same pool policy.
* **Time-aware popularity:** an article's clicks (distinct impression-article
  pairs) in the 24 hours strictly before the request, a window fixed before any
  result. Clicks come from train-week behaviours plus validation-split
  behaviours; the test split's behaviours are never read. "Strictly before"
  excludes the request's own impression. Ties break by recency, then article ID.
  Validation-week clicks come only from validation-split users. This baseline
  uses other users' same-week clicks, which the LSTM never sees.

In [ ]:
POPULARITY_WINDOW_HOURS = 24
run_ranker("most_recent", V.RecencyPopularityRanker(INDEX, ARTICLES))
if not done("popularity_24h", MAIN_SEED):
    click_t, click_ids, click_info = V.load_click_events([PATHS["behaviors_train"], PATHS["behaviors_validation"]])
    run_ranker("popularity_24h", V.RecencyPopularityRanker(INDEX, ARTICLES, click_t, click_ids, POPULARITY_WINDOW_HOURS),
               click_sources=click_info, window_hours=POPULARITY_WINDOW_HOURS)

## 3. Main-run model and recency → LSTM rerank (no training)

The saved main-run model (`model/model_state.pt` and `model_config.json`,
reloaded with the bundled code as in `experiment.ipynb`) is first evaluated on
both samples (`main_model`). The rerank arms then use recency as the candidate
generator: for each catalogue-sample request, take the N newest eligible
articles (the most-recent comparator's order and pool), score only those with
the main-run model, and rank them by its logit. Articles outside the newest N
count as not retrieved. N ∈ {200, 500, 1000} is fixed a priori. Each arm also
records its recall ceiling, Recall@N of the newest N articles. Recorded-slate
metrics are not repeated for the rerank arms: on recorded slates they equal
`main_model`.

In [ ]:
RERANK_N = (200, 500, 1000)
main_model = load_main_model()
if not done("main_model", MAIN_SEED):
    cat, cat_per, slate, slate_per, timing = evaluate_model(main_model, BASE_TENSORS["val"], BASE_TENSORS["slates"])
    save_run("main_model", MAIN_SEED, comparator_metrics(
        "main_model", cat, slate, timing, M.count_parameters(main_model)["trainable_parameters"],
        source=str(MAIN_RUN_DIR / "model")), cat_per, slate_per)
recency = V.RecencyPopularityRanker(INDEX, ARTICLES)
scorer = M.CatalogueScorer(main_model, BASE_TENSORS["val"], INDEX, AGE_STATS, chunk=CATALOGUE_CHUNK, requests=CAT_SAMPLE)
for n in RERANK_N:
    name = f"rerank_{n}"
    if done(name, MAIN_SEED):
        log(f"skip {name}: complete")
        continue
    ceiling, _ = V.evaluate_catalogue(lambda i, k: recency.topk(int(VAL.imp_time_us[i]), k), CAT_SAMPLE, CLICKS_VAL, n)
    cat, cat_per = V.evaluate_catalogue(V.RerankTopK(scorer, recency, INDEX, ARTICLES, n).topk,
                                        CAT_SAMPLE, CLICKS_VAL, K, sync=cuda_sync)
    save_run(name, MAIN_SEED, comparator_metrics(
        name, cat, None, latency(cat), M.count_parameters(main_model)["trainable_parameters"],
        candidates=n, recency_ceiling_recall=ceiling[f"known_click_recall@{n}"]), cat_per)
del main_model, scorer
free_gpu()

## 4. Full model × 3 seeds (noise floor)

Every trained arm uses `V.train_fixed_budget`: the main run's optimiser (AdamW),
AMP, gradient clipping, batch order, history dropout and seeds, for exactly 4
epochs at the tuned constant learning rate, with no scheduler, early stopping or
per-epoch validation evaluation. With seed 42 this replays the main run's first
4 epochs, up to cuDNN/AMP nondeterminism; seed 42 records its Recall@100 on the
main run's selection sample beside the main run's epoch-4 value, as a
reproducibility check only. Each evaluation scores the 5,000 catalogue-sample
requests against their full eligible pools and every recorded-slate candidate.
The seed spread of these three runs is the noise floor.

In [ ]:
for seed in SEEDS:
    run_trained_arm("full", seed)

## 5. History summary: `last_state` × 3 and `attention_no_lstm` × 3

`last_state` replaces target-aware attention with the final LSTM state, the same
for every candidate. `attention_no_lstm` attends directly over the per-event
inputs without recurrence.

In [ ]:
for name in ("last_state", "attention_no_lstm"):
    for seed in SEEDS:
        run_trained_arm(name, seed)

## 6. `metadata_only` (no vectors, no availability flags)

Only about 21,000 of the 125,541 catalogue articles have frozen vectors: those
referenced by this dataset's logs, including the validation week. These inputs
reveal availability: `in_catalogue`, `has_text_vector` and `has_image_vector`
(article numeric columns 6–8); the zero fill (the encoder multiplies each
projected vector by its flag, so a missing vector gives an exactly zero block);
and training-time text dropout, which switches the text flag off for 20% of
candidates. Filling missing vectors with the training-mean vector would not hide
availability: real vectors are unit-norm and their mean is not. So this arm
drops both vector blocks and all three flags, and no input encodes vector
availability. The table shows whether other metadata missingness differs by
vector availability; candidate age is legitimate and stays.

In [ ]:
real = np.arange(1, len(ARTICLES.article_ids))
has_vector = ARTICLES.has_text[real] | ARTICLES.has_image[real]
col = {n: i for i, n in enumerate(NUMERIC_NAMES)}
def availability_row(mask):
    rows = real[mask]
    return {"articles": int(len(rows)),
            "category unknown": float((ARTICLES.category[rows] == 2).mean()),
            "any topic unknown": float((ARTICLES.topics[rows] == 1).any(1).mean()),
            "no topics": float((ARTICLES.topics[rows] == 0).all(1).mean()),
            "body words missing": float(ARTICLES.numeric[rows, col["body_words_missing"]].mean()),
            "sentiment missing": float(ARTICLES.numeric[rows, col["sentiment_missing"]].mean()),
            "premium missing": float(ARTICLES.numeric[rows, col["premium_missing"]].mean()),
            "in_catalogue values": str(np.unique(ARTICLES.numeric[rows, col["in_catalogue"]]).tolist()),
            "median publication": str(pd.Timestamp(int(np.median(ARTICLES.published_s[rows])), unit="s").date())}
display(pd.DataFrame({"with a vector": availability_row(has_vector), "vector-less": availability_row(~has_vector)}).T)
run_trained_arm("metadata_only", MAIN_SEED)

## 7. `no_history`

Histories are emptied, so every request takes the model's `empty_history` path.
The two history-derived request-context inputs (log history length, empty flag)
take their empty-history values, so no history information remains.

In [ ]:
run_trained_arm("no_history", MAIN_SEED)

## 8. Logistic regression (recorded slates only)

Refit on all training recorded slates with the `Baseline Testing` feature list
and estimator; its scaler and encoder fit on train only. Its engineered features
exist only for recorded candidates, so this is a **recorded-slate-only
comparison** with no catalogue metrics.

In [ ]:
if not done("logistic_regression", MAIN_SEED):
    t0 = time.time()
    x_train = V.lr_features(FRAMES["train"], ARTICLES, V.history_profiles(PATHS["history_train"], ARTICLES),
                            PREP["device_columns"])
    lr_model = V.make_lr_pipeline().fit(x_train, FRAMES["train"]["clicked"].to_numpy().astype(int))
    fit_seconds = time.time() - t0
    del x_train
    slate_rows, _ = D.subset_requests_frame(FRAMES["validation"], OFFSETS["validation"], SLATE_SAMPLE)
    x_val = V.lr_features(slate_rows, ARTICLES, V.history_profiles(PATHS["history_validation"], ARTICLES),
                          PREP["device_columns"])
    t0 = time.time()
    slate, slate_per = V.slate_metrics(SLATE_ARRAYS.offsets, SLATE_ARRAYS.labels, lr_model.decision_function(x_val))
    estimator = lr_model.named_steps["model"]
    save_run("logistic_regression", MAIN_SEED, comparator_metrics(
        "logistic_regression", None, slate, {"slate_scoring_seconds": time.time() - t0},
        int(estimator.coef_.size + estimator.intercept_.size), fit_seconds,
        scope="recorded-slate-only comparison: engineered features exist only for slate candidates",
        features={"numeric": V.LR_NUMERIC, "categorical": V.LR_CATEGORICAL},
        solver_iterations=int(estimator.n_iter_[0]), train_rows=int(len(FRAMES["train"]))), slates=slate_per)

## 9. Evaluation-time history permutation (main-run model, no training)

The saved main-run model scores validation with each user's retained events in a
fixed random order (seed 0); each event keeps its chronologically computed
features. Its reference is `main_model` (section 3).

In [ ]:
PERMUTE_SEED = 0
if not done("permute_eval", MAIN_SEED):
    main_model = load_main_model()
    permuted = D.permute_history(VAL, PERMUTE_SEED)
    cat, cat_per, slate, slate_per, timing = evaluate_model(
        main_model, M.SplitTensors(permuted, DEVICE),
        M.SplitTensors(D.subset_split_arrays(permuted, SLATE_SAMPLE), DEVICE))
    save_run("permute_eval", MAIN_SEED, comparator_metrics(
        "permute_eval", cat, slate, timing, M.count_parameters(main_model)["trainable_parameters"],
        permute_seed=PERMUTE_SEED), cat_per, slate_per)

## 10. `no_text` and `no_image`

Each arm removes one vector block. Its availability flag stays, so these arms
test vector content only.

In [ ]:
for name in ("no_text", "no_image"):
    run_trained_arm(name, MAIN_SEED)

## 11. Seed ensemble

Mean of the three full-model seeds' logits, chunk by chunk over each full pool
and per recorded candidate. Logits are used rather than ranks: the seeds share
architecture and training, so their logit scales are comparable, and the mean
keeps score margins that rank averaging discards. The cell reuses section 4's
saved weights and trains any missing seed.

In [ ]:
if not done("ensemble", ENSEMBLE_LABEL):
    for seed in SEEDS:
        run_trained_arm("full", seed)
    members = [load_arm_model("full", s) for s in SEEDS]
    cat, cat_per, slate, slate_per, timing = evaluate_model(None, BASE_TENSORS["val"], BASE_TENSORS["slates"],
                                                            members=members)
    save_run("ensemble", ENSEMBLE_LABEL, comparator_metrics(
        "ensemble", cat, slate, timing, int(sum(M.count_parameters(m)["trainable_parameters"] for m in members)),
        score="mean logit over seeds " + ENSEMBLE_LABEL), cat_per, slate_per)

## 12. Tier 2: history length 10

Each user keeps the most recent 10 of the retained 50 events, in training and evaluation. The first kept event's gap-to-previous is marked missing, as at the main 50-event cut. Requests, candidates and request context stay identical.

In [ ]:
run_trained_arm("history_10", MAIN_SEED)

## 13. Tier 2: history length 20

As section 12, keeping the most recent 20 events.

In [ ]:
run_trained_arm("history_20", MAIN_SEED)

## 14. Tier 2: no per-event read features

Removes read time and scroll (value and missing flag) from each history event; the gap and age-at-read features stay.

In [ ]:
run_trained_arm("no_read_features", MAIN_SEED)

## 15. Tier 2: ID embedding on

Learned article-ID buckets for IDs with at least 5 training references (recorded slates plus retained histories); other IDs share bucket 0.

In [ ]:
ID_MIN_COUNT = 5
hist = HISTORIES["train"]
retained = hist.article_ids[np.arange(hist.article_ids.shape[1])[None, :] < hist.lengths[:, None]]
ID_VOCAB = D.fit_id_vocab(np.concatenate([FRAMES["train"]["candidate_article_id"].to_numpy(), retained]), ID_MIN_COUNT)
ARMS["id_embedding"].update(model=dict(use_id_embedding=True, n_id_buckets=len(ID_VOCAB) + 1),
                            article_tensors=V.with_id_buckets(ARTICLE_TENSORS, ARTICLES, ID_VOCAB, DEVICE))
log("ID vocabulary:", len(ID_VOCAB), "train article IDs")
run_trained_arm("id_embedding", MAIN_SEED)

## 16. Tier 2: no text dropout

Training-time candidate text dropout 0.2 → 0.

In [ ]:
run_trained_arm("no_text_dropout", MAIN_SEED)

## 17. Tier 2: train-time shuffle

Training and validation histories use a fixed permutation (seed 0); each event keeps its chronologically computed features, so only the order the LSTM sees changes.

In [ ]:
ARMS["train_shuffled"]["permute_seed"] = 0
run_trained_arm("train_shuffled", MAIN_SEED)

## 18. Tier 2: no recency term

Removes the learned recency bias from attention (fixed at 0).

In [ ]:
run_trained_arm("no_recency", MAIN_SEED)

## 19. Tier 2: learning curve, 25% of training requests

The 25% and 50% subsets are prefixes of one BLAKE2b order of training requests (label `train_learning_curve`), so 25% is inside 50%. Fitted preprocessing stays the main run's, so only the training rows change; 4 epochs over fewer requests means fewer steps.

In [ ]:
tr = ARRAYS["train"]
ARMS["train_25pct"]["train_requests"] = V.hash_sample(tr.user_id, tr.impression_id, tr.imp_time_us,
    "train_learning_curve", SAMPLE_SEED, int(round(0.25 * tr.n_impressions)))[0]
run_trained_arm("train_25pct", MAIN_SEED)

## 20. Tier 2: learning curve, 50% of training requests

Same BLAKE2b order as section 19.

In [ ]:
tr = ARRAYS["train"]
ARMS["train_50pct"]["train_requests"] = V.hash_sample(tr.user_id, tr.impression_id, tr.imp_time_us,
    "train_learning_curve", SAMPLE_SEED, int(round(0.5 * tr.n_impressions)))[0]
run_trained_arm("train_50pct", MAIN_SEED)

## 21. Results

Reads every arm saved under this `RUN_ID` (any arm in `ARMS` or `COMPARATORS`;
missing runs are skipped) and runs after setup alone. Δ and its CI appear only
where the reference run exists; otherwise the row shows absolute metrics. The
"differs" column applies rule 5. These are validation development results on
samples that exclude the final rerun's selection requests. Sections 22–24 each
depend only on setup and this cell.

In [ ]:
def load_run(name, seed):
    d = arm_dir(name, seed)
    if not (d / "metrics.json").is_file():
        return None
    run = {"metrics": json.loads((d / "metrics.json").read_text())}
    for part in ("catalogue", "slates"):
        path = d / f"{part}.parquet"
        run[part] = pd.read_parquet(path) if path.is_file() else None
    return run

SPECS = {**{n: {**s, "reference": "matched"} for n, s in ARMS.items()}, **COMPARATORS}
ORDER = ["most_recent", "popularity_24h", "main_model", "rerank_200", "rerank_500", "rerank_1000", "full",
         "last_state", "attention_no_lstm", "metadata_only", "no_history", "logistic_regression", "permute_eval",
         "no_text", "no_image", "ensemble", "history_10", "history_20", "no_read_features", "id_embedding",
         "no_text_dropout", "train_shuffled", "no_recency", "train_25pct", "train_50pct"]
ORDER = [n for n in ORDER if n in SPECS] + [n for n in SPECS if n not in ORDER]
RUNS = {n: {s: r for s in SPECS[n]["seeds"] if (r := load_run(n, s)) is not None} for n in ORDER}
FULL = RUNS["full"]
full_recall = {s: r["metrics"]["catalogue"]["known_click_recall@100"] for s, r in FULL.items()}
NOISE_FLOOR = (max(full_recall.values()) - min(full_recall.values())) if len(full_recall) == len(SEEDS) else None
log("Full-model Recall@100 by seed:", full_recall, "| noise floor (seed spread):", NOISE_FLOOR)

def reference_runs(name, runs):
    ref = SPECS[name]["reference"]
    if isinstance(ref, str) and ref != "matched":
        return list(RUNS[ref].values()) or None
    seeds = list(runs) if ref == "matched" else ref
    return [FULL[s] for s in seeds] if all(s in FULL for s in seeds) else None

def reference_label(name):
    ref = SPECS[name]["reference"]
    return ref if isinstance(ref, str) and ref != "matched" else "full"

def delta(name, part, column, mask=None, values=None):
    runs = RUNS[name]
    ref = reference_runs(name, runs)
    if not runs or ref is None or any(r[part] is None for r in list(runs.values()) + ref):
        return None
    get = values or (lambda r: r[part][column].to_numpy())
    users = CAT_USERS if part == "catalogue" else SLATE_USERS
    keep = slice(None) if mask is None else mask
    return V.paired_delta(users[keep], [get(r)[keep] for r in runs.values()],
                          [get(r)[keep] for r in ref], N_BOOT, BOOT_SEED)

fmt = lambda d: None if not d else f"{d['mean_difference']:+.4f} [{d['ci95'][0]:+.4f}, {d['ci95'][1]:+.4f}]"

def mean_metric(runs, *path):
    values = []
    for r in runs.values():
        value = r["metrics"]
        for key in path:
            value = (value or {}).get(key)
        values.append(value)
    values = [v for v in values if v is not None]
    return float(np.mean(values)) if values else None

ROWS = []
for name, runs in RUNS.items():
    if not runs:
        continue
    spec = SPECS[name]
    d = None if name == "full" else delta(name, "catalogue", "recall@100")
    flag = V.differs(d, NOISE_FLOOR)
    ROWS.append({
        "arm": name, "group": spec["group"], "tier": spec["tier"], "change": spec["change"],
        "seeds": ", ".join(map(str, runs)), "reference": None if name == "full" else reference_label(name),
        "Δ Recall@100 [95% CI]": (f"seed spread {NOISE_FLOOR:.4f}" if name == "full" and NOISE_FLOOR is not None
                                  else fmt(d)),
        "differs (rule 5)": None if flag is None else ("yes" if flag else "no"),
        "Recall@100": mean_metric(runs, "catalogue", "known_click_recall@100"),
        "nDCG@100": mean_metric(runs, "catalogue", "observed_click_ndcg@100"),
        "recency ceiling": mean_metric(runs, "recency_ceiling_recall"),
        "Δ slate AUC [95% CI]": None if name == "full" else fmt(delta(name, "slates", "auc")),
        "slate AUC": mean_metric(runs, "recorded_slates", "mean_impression_auc"),
        "slate nDCG@5": mean_metric(runs, "recorded_slates", "mean_impression_ndcg@5"),
        "slate MRR": mean_metric(runs, "recorded_slates", "mean_impression_mrr"),
        "parameters": mean_metric(runs, "parameters"),
        "train s": mean_metric(runs, "train_seconds"),
        "ms / request": mean_metric(runs, "timing", "catalogue_ms_per_request"),
        "ms / 1k candidates": mean_metric(runs, "timing", "catalogue_ms_per_1k_candidates"),
    })
RESULTS = pd.DataFrame(ROWS)
D.write_json({"noise_floor": NOISE_FLOOR, "full_recall_by_seed": full_recall,
              "rows": RESULTS.to_dict("records")}, ABLATION_DIR / "results_table.json")
pd.set_option("display.max_colwidth", 80)
display(RESULTS.round(4))

## 22. Segments

Δ Recall@100 against each arm's reference (95% CI) by the user's complete history
length before truncation, and by whether a clicked article is cold (never in any
training recorded slate or any event of the complete training history file) or
warm. "requests" counts requests with a known click in the segment.

In [ ]:
train_history_ids = pq.read_table(PATHS["history_train"], columns=["article_id_fixed"])["article_id_fixed"]
train_history_ids = train_history_ids.combine_chunks().flatten().drop_null().to_numpy().astype(np.int64)
WARM_IDS = np.union1d(FRAMES["train"]["candidate_article_id"].to_numpy(), train_history_ids)

SEGMENT_ARMS = [a for a in ["most_recent", "rerank_500", "last_state", "attention_no_lstm", "metadata_only",
                            "no_history", "permute_eval", "no_text", "no_image", "popularity_24h"] if RUNS.get(a)]
CAT_HISTORY_BUCKET = V.history_bucket(VAL.hist_full_lengths[VAL.imp_user_row[CAT_SAMPLE]])
_cold_warm = {}
def cold_warm(run, which):
    if id(run) not in _cold_warm:
        _cold_warm[id(run)] = V.cold_warm_recall(run["catalogue"]["top100"].tolist(), CAT_SAMPLE,
                                                 CLICK_LOOKUP, WARM_IDS, K)
    return _cold_warm[id(run)][which]

segments = [(f"history {b}", CAT_HISTORY_BUCKET == b, None) for _, _, b in V.HISTORY_BUCKETS] + \
           [("cold clicked article", None, 0), ("warm clicked article", None, 1)]
SEGMENT_ROWS = []
for label, mask, which in segments:
    values = None if which is None else (lambda r, w=which: cold_warm(r, w))
    base = np.vstack([values(r) if values else r["catalogue"]["recall@100"].to_numpy() for r in FULL.values()]).mean(0) \
        if FULL else np.full(len(CAT_SAMPLE), np.nan)
    base = base if mask is None else base[mask]
    row = {"segment": label, "requests": int(np.isfinite(base).sum()),
           "full Recall@100": float(np.nanmean(base)) if np.isfinite(base).any() else None}
    for name in SEGMENT_ARMS:
        row[f"{name} vs {reference_label(name)}"] = fmt(delta(name, "catalogue", "recall@100", mask=mask, values=values)) \
            if row["requests"] else None
    SEGMENT_ROWS.append(row)
SEGMENTS = pd.DataFrame(SEGMENT_ROWS)
D.write_json(SEGMENTS.to_dict("records"), ABLATION_DIR / "segment_table.json")
display(SEGMENTS)

## 23. Vector share

Mean share of vector-less articles (no text and no image vector) in each arm's
top 100, against the share in the full eligible pool and in the fresh pool
(published at most 72 hours before the request), averaged over the catalogue
sample. The known-click row shows the share among clicked articles.

In [ ]:
FRESH_POOL_HOURS = 72
POOL_SHARE = V.pool_vectorless_share(INDEX, ARTICLES, VAL.imp_time_us[CAT_SAMPLE], FRESH_POOL_HOURS)
clicked_share = [float((~(ARTICLES.has_text[r] | ARTICLES.has_image[r])).mean())
                 for i in CAT_SAMPLE if (e := CLICK_LOOKUP.get(int(i))) is not None
                 for r in [ARTICLES.rows_for(e[0])]]
fresh = float(np.nanmean(POOL_SHARE["fresh_pool"]))
share_rows = [{"ranking": "eligible pool (all)", "vector-less share": float(np.mean(POOL_SHARE["full_pool"]))},
              {"ranking": f"fresh pool (<= {FRESH_POOL_HOURS} h; median size "
                          f"{int(np.median(POOL_SHARE['fresh_pool_size']))})", "vector-less share": fresh},
              {"ranking": "known clicks", "vector-less share": float(np.mean(clicked_share))}]
for name, runs in RUNS.items():
    if runs and all(r["catalogue"] is not None for r in runs.values()):
        share = np.mean([np.nanmean(V.top_vectorless_share(r["catalogue"]["top100"].tolist(), ARTICLES, K))
                         for r in runs.values()])
        share_rows.append({"ranking": f"{name} top {K}", "vector-less share": float(share)})
VECTOR_SHARE = pd.DataFrame(share_rows)
VECTOR_SHARE["ratio to fresh pool"] = VECTOR_SHARE["vector-less share"] / fresh
D.write_json(VECTOR_SHARE.to_dict("records"), ABLATION_DIR / "vector_share_table.json")
display(VECTOR_SHARE.round(4))

## 24. Ensemble row and main-run training curves

The ensemble row repeats the results-table entry (Δ against the mean of the three
single full models). The curves come from the main run's saved
`training_history.json`; nothing is retrained. They show the selection sample's
Recall@100, so they are a development curve, not a reported ablation result.

In [ ]:
display(RESULTS[RESULTS["arm"].isin(["full", "ensemble"])].round(4))
history = pd.DataFrame(MAIN_HISTORY)
fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
for ax, (column, title) in zip(axes, [("train_bce", "Training BCE"),
                                      ("known_click_recall@100", "Selection-sample Recall@100"),
                                      ("diagnostic_recorded_slate_auc", "Selection-sample slate AUC")]):
    ax.plot(history["epoch"], history[column], marker="o")
    ax.axvline(EPOCHS, color="grey", linestyle="--", linewidth=1)
    ax.set_title(title)
    ax.set_xlabel("epoch")
fig.suptitle(f"Main run {MAIN_RUN_ID}; dashed line = fixed ablation budget ({EPOCHS} epochs)")
fig.tight_layout()
plt.show()

## 25. Reading the results

Fill this in from the tables above, applying the pre-registered rules. Write
"no detectable difference" when rule 5 fails; do not claim equality.

* **Noise floor** (full-model seed spread on Recall@100): ___.
* **Baselines (rules 5–6):** full − most recent = ___; full − popularity = ___;
  main-run model − full seed 42 = ___. Full model below most recent: yes / no
  (if yes, that is the headline). Logistic regression (recorded slates only):
  Δ slate AUC = ___; no catalogue claim.
* **LSTM on top of recency (rule 7):** rerank − most recent at N = 200 / 500 / 1000:
  ___ / ___ / ___, against ceilings ___ / ___ / ___. The LSTM adds value as a
  recency reranker: yes / no.
* **Attention (rule 8):** Δ(`last_state`) = ___ [___, ___]. Attention earns its
  cost: yes / no. Parameter and latency difference: ___.
* **Recurrence (rule 8):** Δ(`attention_no_lstm`) = ___ [___, ___]. The LSTM
  earns its cost: yes / no.
* **Order (rule 10):** Δ(`permute_eval` vs `main_model`) = ___; Δ(`train_shuffled`,
  tier 2) = ___. The model uses event order: yes / no.
* **History:** Δ(`no_history`) = ___. Long-history framing (rule 9), from the
  segment table's 50+ buckets: used / not used.
* **Exposure shortcut (rule 11):** vector-less share in the full model's top 100 =
  ___ vs fresh pool ___ vs `metadata_only` ___. Δ(`metadata_only`) = ___.
* **Content vectors:** Δ(`no_text`) = ___; Δ(`no_image`) = ___.
* **Cold start:** cold vs warm clicked-article segments for the content arms: ___.
* **Ensemble:** Δ vs a single model = ___ at 3× the parameters and scoring time.
* **Tier 2** (if run): history length ___; read features ___; ID embedding ___;
  text dropout ___; recency ___; learning curve ___.
* **Limitations:**
  * Validation development samples; observed-click recall under exposure bias;
    unshown articles have unknown outcomes; one training budget; single seeds
    outside the full model and group B.
  * `no_text` and `no_image` keep their availability flag, so they test vector
    content only.
  * Vectors exist only for logged articles, so the main model may benefit from
    this vector-availability exposure leak; `metadata_only` is the arm without it.
  * Single-seed arms built with `AblationRanker` (`no_text`, `no_image`,
    `metadata_only`, `no_read_features`, `no_recency`) are not
    initialisation-matched with full seed 42, because the class builds and then
    replaces the article encoder; the seed spread covers this variation.
  * `main_model`, the rerank arms and `permute_eval` use the saved main-run
    checkpoint, which differs from the retrained full seed 42 by training
    nondeterminism; the `main_model` row shows the size of that gap.